In [ ]:
# ============================================================
# NOTE 4 – ANOMALY & EVENT DETECTION
# RQ3: Những biến động bất thường nào xuất hiện trong quá trình chuyển dịch nguồn điện?
# ============================================================

import pandas as pd
from pathlib import Path

relative_path = Path("checkpoints/note03_eda_overview.pkl")

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / relative_path).is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        f"Không tìm thấy file: {relative_path}"
    )
df_path = project_root / relative_path

note03_eda_overview = (
    project_root / "checkpoints/note03_eda_overview.pkl"
)

note03_state = pd.read_pickle(note03_eda_overview)

df = note03_state["df"]
generation_sources = note03_state["generation_sources"]
system_sources = note03_state["system_sources"]
source_summary = note03_state["source_summary"]
global_trend = note03_state["global_trend"]
global_sources = note03_state["global_sources"]
country_core_coverage = note03_state["country_core_coverage"]
country_panel = note03_state["country_panel"]
country_source_panel = note03_state["country_source_panel"]
wind_coverage = note03_state["wind_coverage"]
country_generation_2022 = note03_state["country_generation_2022"]
country_mix_2022 = note03_state["country_mix_2022"]
country_mix_check = note03_state["country_mix_check"]
mix_balance = note03_state["mix_balance"]
nuclear_2022 = note03_state["nuclear_2022"]

print("=" * 65)
print("NOTE 4 – ANOMALY & EVENT DETECTION")
print("=" * 65)
print("Đã khôi phục trạng thái từ Note 3 – EDA Tổng quan.")
print("Note 4 sẽ tập trung trả lời RQ3:")
print("Những biến động bất thường nào xuất hiện trong quá trình chuyển dịch nguồn điện?")
print()
print("Dataset shape:", df.shape)
print("Year range:", df["Year"].min(), "→", df["Year"].max())
print("Country/economy:", (df["Area type"] == "Country or economy").sum())

NOTE 4 – ANOMALY & EVENT DETECTION
Đã khôi phục trạng thái từ Note 3 – EDA Tổng quan.
Note 4 sẽ tập trung trả lời RQ3:
Những biến động bất thường nào xuất hiện trong quá trình chuyển dịch nguồn điện?

Dataset shape: (104203, 24)
Year range: 1985 → 2025
Country/economy: 97937


In [3]:
# ============================================================
# Ở cấp quốc gia, những năm nào có thay đổi lớn bất thường
# về tỷ trọng của các nguồn điện so với năm trước?
#
# Mục đích:
# - Tạo danh sách candidate anomaly một cách có hệ thống.
# - Không chọn event trước khi xem dữ liệu.
# - Chỉ xét các nguồn phát điện chính, không dùng:
#   + Total generation
#   + Demand
#   + Net imports
#   + các dòng aggregate để tránh double counting.
# ============================================================

# Các nguồn thành phần thực sự dùng để phân tích anomaly
anomaly_sources = [
    "Solar",
    "Wind",
    "Hydro",
    "Bioenergy",
    "Other renewables",
    "Nuclear",
    "Coal",
    "Gas",
    "Other fossil"
]

# Chỉ lấy Country or economy
anomaly_df = df[
    (df["Area type"] == "Country or economy") &
    (df["Electricity source"].isin(anomaly_sources))
].copy()

# Sắp xếp để YoY change có đúng thứ tự thời gian
anomaly_df = anomaly_df.sort_values(
    ["Area", "Electricity source", "Year"]
)

# Tính thay đổi tỷ trọng theo năm
anomaly_df["share_change_pp"] = (
    anomaly_df
    .groupby(["Area", "Electricity source"])["Share of generation (%)"]
    .diff()
)

# Chỉ giữ các biến động đủ lớn để làm candidate
candidate_anomalies = (
    anomaly_df[
        anomaly_df["share_change_pp"].abs() >= 5
    ]
    .sort_values(
        "share_change_pp",
        key=lambda x: x.abs(),
        ascending=False
    )
)

print("Đã quét biến động tỷ trọng nguồn điện theo từng quốc gia.")
print()
print("Tiêu chí screening:")
print("- Country or economy")
print("- 9 nguồn phát điện thành phần")
print("- |Thay đổi tỷ trọng YoY| >= 5 percentage points")
print()
print("Số candidate anomalies:", len(candidate_anomalies))
print()
print("Top 30 candidate anomalies:")
print(
    candidate_anomalies[
        [
            "Area",
            "Year",
            "Electricity source",
            "Share of generation (%)",
            "share_change_pp",
            "Generation (TWh)"
        ]
    ]
    .head(30)
    .to_string(index=False)
)

Đã quét biến động tỷ trọng nguồn điện theo từng quốc gia.

Tiêu chí screening:
- Country or economy
- 9 nguồn phát điện thành phần
- |Thay đổi tỷ trọng YoY| >= 5 percentage points

Số candidate anomalies: 2559

Top 30 candidate anomalies:
                       Area  Year Electricity source  Share of generation (%)  share_change_pp  Generation (TWh)
                    Germany  1990                Gas                    6.620          -93.380             35.90
                      Malta  2017                Gas                   78.182           78.182              1.29
                  Lithuania  2010            Nuclear                    0.000          -74.112              0.00
                      Malta  2017       Other fossil                   11.515          -72.206              0.19
       Macao (SAR of China)  2015       Other fossil                   78.947           58.635              0.75
               Sierra Leone  2010       Other fossil                   14.286      

In [5]:
# ============================================================
# Những anomaly lớn nào vẫn đáng chú ý khi quốc gia có quy mô
# sản xuất điện đủ lớn?
#
# Mục đích:
# Loại các biến động phần trăm bị khuếch đại bởi quy mô điện
# quá nhỏ, nhưng chưa kết luận đây là event thực tế.
# ============================================================

# Lấy Total generation làm quy mô hệ thống của từng quốc gia-năm
total_generation = (
    df[
        (df["Area type"] == "Country or economy") &
        (df["Electricity source"] == "Total generation")
    ][
        ["Area", "Year", "Generation (TWh)"]
    ]
    .rename(columns={"Generation (TWh)": "Total_generation_TWh"})
)

# Ghép quy mô hệ thống vào candidate anomalies
candidate_anomalies_scaled = candidate_anomalies.merge(
    total_generation,
    on=["Area", "Year"],
    how="left"
)

# Chỉ giữ quốc gia-năm có tổng sản lượng >= 100 TWh
candidate_large_system = (
    candidate_anomalies_scaled[
        candidate_anomalies_scaled["Total_generation_TWh"] >= 100
    ]
    .sort_values(
        "share_change_pp",
        key=lambda x: x.abs(),
        ascending=False
    )
)

print("Đã bổ sung quy mô sản xuất điện của từng quốc gia-năm.")
print()
print("Tiêu chí lọc:")
print("- |Thay đổi tỷ trọng YoY| >= 5 percentage points")
print("- Total generation >= 100 TWh")
print()
print("Số candidate sau khi lọc:", len(candidate_large_system))
print()
print("Top 30 candidate anomalies có quy mô lớn:")
print(
    candidate_large_system[
        [
            "Area",
            "Year",
            "Electricity source",
            "Share of generation (%)",
            "share_change_pp",
            "Generation (TWh)",
            "Total_generation_TWh"
        ]
    ]
    .head(30)
    .to_string(index=False)
)

Đã bổ sung quy mô sản xuất điện của từng quốc gia-năm.

Tiêu chí lọc:
- |Thay đổi tỷ trọng YoY| >= 5 percentage points
- Total generation >= 100 TWh

Số candidate sau khi lọc: 256

Top 30 candidate anomalies có quy mô lớn:
                Area  Year Electricity source  Share of generation (%)  share_change_pp  Generation (TWh)  Total_generation_TWh
             Germany  1990                Gas                    6.620          -93.380             35.90                542.26
                Iran  2014                Gas                   69.659           16.468            192.00                275.63
                Iraq  2023       Other fossil                   46.597           16.274             67.57                145.01
                Iran  2014       Other fossil                   23.321          -15.924             64.28                275.63
                Iraq  2023                Gas                   52.190          -15.224             75.68                145.01
         

In [7]:
# ============================================================
# Một anomaly lớn có phản ánh thay đổi thực sự trong cơ cấu
# sản xuất điện hay chỉ là structural/data break?
#
# Mục đích:
# - Lấy 15 candidate mạnh nhất.
# - So sánh cơ cấu nguồn giữa năm t-1 và năm t.
# - Kiểm tra Total generation có biến động bất thường không.
# - Tạo cơ sở để loại data artifact trước khi chọn event.
# ============================================================

# 15 candidate mạnh nhất theo độ lớn của thay đổi tỷ trọng
top_candidates = (
    candidate_large_system
    .assign(abs_change=lambda x: x["share_change_pp"].abs())
    .sort_values("abs_change", ascending=False)
    .drop(columns="abs_change")
    .head(15)
    .copy()
)

# Danh sách quốc gia-năm cần kiểm tra
candidate_pairs = top_candidates[
    ["Area", "Year"]
].drop_duplicates()

# Lấy dữ liệu cơ cấu nguồn của năm anomaly và năm trước
validation_rows = []

for _, row in candidate_pairs.iterrows():
    area = row["Area"]
    year = row["Year"]

    temp = anomaly_df[
        (anomaly_df["Area"] == area) &
        (anomaly_df["Year"].isin([year - 1, year]))
    ].copy()

    if not temp.empty:
        temp["candidate_year"] = year
        validation_rows.append(temp)

candidate_validation = pd.concat(
    validation_rows,
    ignore_index=True
)

# Pivot tỷ trọng để so sánh trực tiếp năm trước và năm anomaly
validation_pivot = (
    candidate_validation
    .pivot_table(
        index=["Area", "candidate_year", "Electricity source"],
        columns="Year",
        values="Share of generation (%)",
        aggfunc="first"
    )
    .reset_index()
)

# Đổi tên hai cột năm thành rõ nghĩa
validation_pivot = validation_pivot.rename(
    columns=lambda x: (
        "share_previous_year"
        if isinstance(x, (int, float)) and x < validation_pivot["candidate_year"].iloc[0]
        else (
            "share_event_year"
            if isinstance(x, (int, float))
            else x
        )
    )
)


print("15 candidate mạnh nhất theo |share_change_pp|:")
print(
    top_candidates[
        [
            "Area",
            "Year",
            "Electricity source",
            "share_change_pp",
            "Generation (TWh)",
            "Total_generation_TWh"
        ]
    ].to_string(index=False)
)

print()
print("=" * 65)
print("CƠ CẤU NGUỒN CỦA CÁC CANDIDATE")
print("=" * 65)

for _, row in candidate_pairs.iterrows():
    area = row["Area"]
    year = row["Year"]

    print(f"\n--- {area}: {year-1} → {year} ---")

    check = candidate_validation[
        (candidate_validation["Area"] == area) &
        (candidate_validation["Year"].isin([year - 1, year]))
    ][
        ["Year", "Electricity source", "Share of generation (%)"]
    ].copy()

    print(
        check
        .sort_values(["Year", "Share of generation (%)"],
                     ascending=[True, False])
        .to_string(index=False)
    )

15 candidate mạnh nhất theo |share_change_pp|:
                Area  Year Electricity source  share_change_pp  Generation (TWh)  Total_generation_TWh
             Germany  1990                Gas          -93.380             35.90                542.26
                Iran  2014                Gas           16.468            192.00                275.63
                Iraq  2023       Other fossil           16.274             67.57                145.01
                Iran  2014       Other fossil          -15.924             64.28                275.63
                Iraq  2023                Gas          -15.224             75.68                145.01
                Iran  2011                Gas          -13.351            142.33                235.59
      United Kingdom  2016               Coal          -13.350             30.67                339.18
               Japan  2012            Nuclear          -13.130             17.99               1106.88
                Iran  2011

In [8]:
# ============================================================
# Event nào có thay đổi đủ lớn cả về tỷ trọng và sản lượng?
#
# Mục đích:
# Không chọn event chỉ dựa trên % thay đổi.
# Ta xem đồng thời:
# 1. Share change (percentage points)
# 2. Generation change (TWh)
# 3. Quy mô tổng sản xuất điện
# ============================================================

# Các event đáng xem xét sau bước validation
event_candidates = [
    ("United Kingdom", 2016),
    ("Japan", 2012),
    ("Netherlands", 2021),
    ("Türkiye", 2019),
    ("United Arab Emirates", 2023)
]

event_results = []

for area, year in event_candidates:

    # Dữ liệu năm trước và năm event
    temp = df[
        (df["Area"] == area) &
        (df["Area type"] == "Country or economy") &
        (df["Electricity source"].isin(anomaly_sources)) &
        (df["Year"].isin([year - 1, year]))
    ].copy()

    # Pivot generation
    gen_pivot = temp.pivot_table(
        index="Electricity source",
        columns="Year",
        values="Generation (TWh)",
        aggfunc="first"
    )

    # Pivot share
    share_pivot = temp.pivot_table(
        index="Electricity source",
        columns="Year",
        values="Share of generation (%)",
        aggfunc="first"
    )

    # Chỉ lấy nguồn có đủ cả 2 năm
    common_sources = gen_pivot.dropna().index.intersection(
        share_pivot.dropna().index
    )

    for source in common_sources:
        gen_change = (
            gen_pivot.loc[source, year]
            - gen_pivot.loc[source, year - 1]
        )

        share_change = (
            share_pivot.loc[source, year]
            - share_pivot.loc[source, year - 1]
        )

        event_results.append({
            "Area": area,
            "Year": year,
            "Source": source,
            "Generation_previous_TWh": gen_pivot.loc[source, year - 1],
            "Generation_event_TWh": gen_pivot.loc[source, year],
            "Generation_change_TWh": gen_change,
            "Share_change_pp": share_change
        })

event_comparison = pd.DataFrame(event_results)

# Sắp xếp theo độ lớn thay đổi tỷ trọng
event_comparison["abs_share_change_pp"] = (
    event_comparison["Share_change_pp"].abs()
)

event_comparison = event_comparison.sort_values(
    ["Area", "abs_share_change_pp"],
    ascending=[True, False]
)


print("Các event được đưa vào so sánh:")
print(
    ", ".join(
        [f"{area} {year}" for area, year in event_candidates]
    )
)

print()
print("Thay đổi nguồn điện của từng event:")
print()

for area, year in event_candidates:

    print(f"--- {area}: {year-1} → {year} ---")

    result = event_comparison[
        (event_comparison["Area"] == area) &
        (event_comparison["Year"] == year)
    ][
        [
            "Source",
            "Generation_previous_TWh",
            "Generation_event_TWh",
            "Generation_change_TWh",
            "Share_change_pp"
        ]
    ]

    print(
        result.to_string(index=False)
    )
    print()

Các event được đưa vào so sánh:
United Kingdom 2016, Japan 2012, Netherlands 2021, Türkiye 2019, United Arab Emirates 2023

Thay đổi nguồn điện của từng event:

--- United Kingdom: 2015 → 2016 ---
          Source  Generation_previous_TWh  Generation_event_TWh  Generation_change_TWh  Share_change_pp
            Coal                    75.88                 30.67                 -45.21          -13.350
             Gas                    99.88                143.13                  43.25           12.725
            Wind                    40.27                 37.16                  -3.11           -0.928
           Solar                     7.53                 10.41                   2.88            0.847
         Nuclear                    70.34                 71.73                   1.39            0.391
    Other fossil                     9.41                 10.64                   1.23            0.360
           Hydro                     6.30                  5.37            

In [9]:
# ============================================================
# Sau anomaly năm 2012, cơ cấu điện của Japan có thực sự
# thay đổi và duy trì trong các năm tiếp theo không?
#
# Mục đích:
# Phân biệt một shock ngắn hạn với một thay đổi cơ cấu
# có tính duy trì.
# ============================================================

japan_event = df[
    (df["Area"] == "Japan") &
    (df["Area type"] == "Country or economy") &
    (df["Electricity source"].isin(anomaly_sources)) &
    (df["Year"].between(2008, 2016))
].copy()

japan_event = japan_event[
    [
        "Year",
        "Electricity source",
        "Generation (TWh)",
        "Share of generation (%)"
    ]
].sort_values(
    ["Year", "Share of generation (%)"],
    ascending=[True, False]
)

# Bảng riêng cho các nguồn quan trọng nhất
japan_key_sources = japan_event[
    japan_event["Electricity source"].isin(
        ["Nuclear", "Gas", "Coal", "Other fossil", "Hydro", "Solar", "Wind"]
    )
].copy()

japan_key_pivot = (
    japan_key_sources
    .pivot_table(
        index="Year",
        columns="Electricity source",
        values="Share of generation (%)",
        aggfunc="first"
    )
    .reset_index()
)


print("Theo dõi Japan trong giai đoạn 2008–2016.")
print("Mục tiêu: kiểm tra anomaly 2012 có kéo dài sang các năm sau hay không.")
print()
print("Tỷ trọng các nguồn chính (%):")
print(
    japan_key_pivot.to_string(index=False)
)

print()
print("Sản lượng Nuclear và Gas (TWh):")

japan_generation_check = (
    japan_key_sources[
        japan_key_sources["Electricity source"].isin(
            ["Nuclear", "Gas"]
        )
    ]
    .pivot_table(
        index="Year",
        columns="Electricity source",
        values="Generation (TWh)",
        aggfunc="first"
    )
    .reset_index()
)

print(
    japan_generation_check.to_string(index=False)
)

Theo dõi Japan trong giai đoạn 2008–2016.
Mục tiêu: kiểm tra anomaly 2012 có kéo dài sang các năm sau hay không.

Tỷ trọng các nguồn chính (%):
 Year   Coal    Gas  Hydro  Nuclear  Other fossil  Solar  Wind
 2008 28.191 27.442  6.350   21.267        14.484  0.219 0.248
 2009 28.014 29.148  6.326   24.654         9.451  0.274 0.308
 2010 27.408 28.208  7.655   25.290         8.866  0.344 0.340
 2011 25.817 34.399  7.471   14.755        14.750  0.493 0.404
 2012 30.202 39.344  6.966    1.625        18.771  0.666 0.427
 2013 33.330 38.941  7.293    1.342        15.305  1.187 0.472
 2014 33.461 42.427  7.689    0.000        11.512  2.216 0.471
 2015 33.942 40.529  8.326    0.439        10.139  3.353 0.507
 2016 31.831 43.098  7.467    1.662         9.141  4.074 0.502

Sản lượng Nuclear và Gas (TWh):
 Year    Gas  Nuclear
 2008 324.83   251.74
 2009 324.72   274.65
 2010 326.09   292.36
 2011 379.83   162.93
 2012 435.49    17.99
 2013 423.58    14.60
 2014 450.88     0.00
 2015 417.50     

In [10]:
# ============================================================
# Sự kiện Japan 2012 đã làm thay đổi cơ cấu sản xuất điện
# như thế nào?
#
# Những nguồn nào thay đổi mạnh nhất về sản lượng và tỷ trọng
# từ trước event đến event và giai đoạn hậu event?
#
# Mục đích:
# Tạo bảng định lượng chính thức cho event Japan 2012.
# ============================================================

japan_before_after = df[
    (df["Area"] == "Japan") &
    (df["Area type"] == "Country or economy") &
    (df["Electricity source"].isin(anomaly_sources)) &
    (df["Year"].isin([2011, 2012, 2016]))
][
    [
        "Year",
        "Electricity source",
        "Generation (TWh)",
        "Share of generation (%)"
    ]
].copy()

japan_before_after = japan_before_after.sort_values(
    ["Electricity source", "Year"]
)

# Pivot Generation
japan_generation_pivot = (
    japan_before_after
    .pivot_table(
        index="Electricity source",
        columns="Year",
        values="Generation (TWh)",
        aggfunc="first"
    )
    .reset_index()
)

# Pivot Share
japan_share_pivot = (
    japan_before_after
    .pivot_table(
        index="Electricity source",
        columns="Year",
        values="Share of generation (%)",
        aggfunc="first"
    )
    .reset_index()
)

# Thay đổi 2011 → 2012
japan_event_change = japan_generation_pivot.copy()

japan_event_change["Generation_change_2011_2012_TWh"] = (
    japan_event_change[2012] -
    japan_event_change[2011]
)

japan_event_change = japan_event_change.merge(
    japan_share_pivot[
        [
            "Electricity source",
            2011,
            2012,
            2016
        ]
    ],
    on="Electricity source",
    suffixes=("_gen", "_share")
)

japan_event_change["Share_change_2011_2012_pp"] = (
    japan_event_change["2012_share"] -
    japan_event_change["2011_share"]
)

japan_event_change = japan_event_change.sort_values(
    "Share_change_2011_2012_pp",
    key=lambda x: x.abs(),
    ascending=False
)

print("Japan event: 2011 → 2012")
print("Post-event reference: 2016")
print()
print("Bảng định lượng:")
print(
    japan_event_change[
        [
            "Electricity source",
            "2011_gen",
            "2012_gen",
            "2016_gen",
            "Generation_change_2011_2012_TWh",
            "2011_share",
            "2012_share",
            "2016_share",
            "Share_change_2011_2012_pp"
        ]
    ].to_string(index=False)
)

Japan event: 2011 → 2012
Post-event reference: 2016

Bảng định lượng:
Electricity source  2011_gen  2012_gen  2016_gen  Generation_change_2011_2012_TWh  2011_share  2012_share  2016_share  Share_change_2011_2012_pp
           Nuclear    162.93     17.99     17.68                          -144.94      14.755       1.625       1.662                    -13.130
               Gas    379.83    435.49    458.43                            55.66      34.399      39.344      43.098                      4.945
              Coal    285.07    334.30    338.59                            49.23      25.817      30.202      31.831                      4.385
      Other fossil    162.87    207.77     97.23                            44.90      14.750      18.771       9.141                      4.021
             Hydro     82.49     77.10     79.43                            -5.39       7.471       6.966       7.467                     -0.505
             Solar      5.44      7.37     43.33            

In [12]:
# ============================================================
#   - Tổng sản lượng điện có thay đổi mạnh không?
#   - Phần điện hạt nhân mất đi được thay thế chủ yếu bởi nhóm nào?
#   - Cơ cấu Nuclear / Fossil / Renewables thay đổi ra sao?
#
# Mục đích:
#   Xác định liệu Japan 2012 là một thay đổi cơ cấu thực sự,
#   thay vì chỉ là biến động của một nguồn điện riêng lẻ.
# ============================================================

japan_event_years = [2011, 2012, 2016]

# Lấy các nhóm nguồn cần thiết
japan_event_df = df[
    (df["Area"] == "Japan") &
    (df["Area type"] == "Country or economy") &
    (df["Year"].isin(japan_event_years)) &
    (df["Electricity source"].isin([
        "Total generation",
        "Nuclear",
        "Fossil",
        "Renewables"
    ]))
].copy()

# Bảng sản lượng
japan_event_generation = (
    japan_event_df
    .pivot(
        index="Electricity source",
        columns="Year",
        values="Generation (TWh)"
    )
    .reindex([
        "Total generation",
        "Nuclear",
        "Fossil",
        "Renewables"
    ])
)

# Bảng tỷ trọng
japan_event_share = (
    japan_event_df
    .pivot(
        index="Electricity source",
        columns="Year",
        values="Share of generation (%)"
    )
    .reindex([
        "Nuclear",
        "Fossil",
        "Renewables"
    ])
)


print("\n1. Sản lượng điện (TWh):")
display(japan_event_generation.round(2))

print("\n2. Tỷ trọng trong tổng sản lượng điện (%):")
display(japan_event_share.round(3))

print("\n3. Thay đổi tỷ trọng 2011 → 2012 (điểm %):")

japan_event_share_change = (
    japan_event_share[2012] - japan_event_share[2011]
).rename("Change_2011_2012_pp")

display(
    japan_event_share_change
    .to_frame()
    .round(3)
)

print("\n→ Dùng kết quả này để xác định nhóm nguồn nào")
print("  đã thay thế phần sản lượng điện hạt nhân bị sụt giảm.")


1. Sản lượng điện (TWh):


Year,2011,2012,2016
Electricity source,,,
Total generation,1104.20,1106.88,1063.70
Nuclear,162.93,17.99,17.68
Fossil,827.77,977.56,894.25
Renewables,113.50,111.33,151.77



2. Tỷ trọng trong tổng sản lượng điện (%):


Year,2011,2012,2016
Electricity source,,,
Nuclear,14.755,1.625,1.662
Fossil,74.966,88.317,84.070
Renewables,10.279,10.058,14.268



3. Thay đổi tỷ trọng 2011 → 2012 (điểm %):


,Change_2011_2012_pp
Electricity source,
Nuclear,-13.130
Fossil,13.351
Renewables,-0.221



→ Dùng kết quả này để xác định nhóm nguồn nào
  đã thay thế phần sản lượng điện hạt nhân bị sụt giảm.


In [14]:
# ============================================================
#   - Lượng Nuclear bị mất đi có được Fossil bù đắp hay không?
#   - Mức tăng của Fossil tương đương bao nhiêu % lượng Nuclear giảm?
#   - Tổng sản lượng điện có thực sự thay đổi đáng kể không?
#
# Mục đích:
#   Định lượng "mức độ thay thế" để chứng minh Japan 2012
#   là một structural break về cơ cấu nguồn điện.
# ============================================================

nuclear_loss = (
    japan_event_generation.loc["Nuclear", 2012]
    - japan_event_generation.loc["Nuclear", 2011]
)

fossil_gain = (
    japan_event_generation.loc["Fossil", 2012]
    - japan_event_generation.loc["Fossil", 2011]
)

renewable_change = (
    japan_event_generation.loc["Renewables", 2012]
    - japan_event_generation.loc["Renewables", 2011]
)

total_generation_change = (
    japan_event_generation.loc["Total generation", 2012]
    - japan_event_generation.loc["Total generation", 2011]
)

replacement_ratio = (
    fossil_gain / abs(nuclear_loss) * 100
)


print("ĐỊNH LƯỢNG MỨC ĐỘ THAY THẾ NUCLEAR → FOSSIL")

print(f"\nNuclear thay đổi: {nuclear_loss:.2f} TWh")
print(f"Fossil thay đổi: {fossil_gain:.2f} TWh")
print(f"Renewables thay đổi: {renewable_change:.2f} TWh")
print(f"Tổng sản lượng thay đổi: {total_generation_change:.2f} TWh")

print(
    f"\nFossil tăng tương đương {replacement_ratio:.2f}% "
    "lượng Nuclear bị mất."
)

print("\n→ Nếu tỷ lệ này xấp xỉ hoặc vượt 100%,")
print("  có thể xem Fossil là nhóm nguồn chính bù đắp")
print("  cho sự sụt giảm của Nuclear trong năm 2012.")

ĐỊNH LƯỢNG MỨC ĐỘ THAY THẾ NUCLEAR → FOSSIL

Nuclear thay đổi: -144.94 TWh
Fossil thay đổi: 149.79 TWh
Renewables thay đổi: -2.17 TWh
Tổng sản lượng thay đổi: 2.68 TWh

Fossil tăng tương đương 103.35% lượng Nuclear bị mất.

→ Nếu tỷ lệ này xấp xỉ hoặc vượt 100%,
  có thể xem Fossil là nhóm nguồn chính bù đắp
  cho sự sụt giảm của Nuclear trong năm 2012.


In [19]:
# ============================================================
#   Trong phần Fossil tăng thêm 149.79 TWh, Gas, Coal và
#   Other fossil đóng góp bao nhiêu?
#
# Mục đích:
#   Xác định nguồn điện cụ thể đứng sau sự thay thế Nuclear.
# ============================================================

japan_fossil_sources = (
    df[
        (df["Area"] == "Japan") &
        (df["Area type"] == "Country or economy") &
        (df["Year"].isin([2011, 2012, 2016])) &
        (df["Electricity source"].isin([
            "Gas",
            "Coal",
            "Other fossil"
        ]))
    ]
    .pivot(
        index="Electricity source",
        columns="Year",
        values="Generation (TWh)"
    )
)

japan_fossil_sources["Change_2011_2012_TWh"] = (
    japan_fossil_sources[2012]
    - japan_fossil_sources[2011]
)

japan_fossil_sources["Contribution_to_fossil_gain_%"] = (
    japan_fossil_sources["Change_2011_2012_TWh"]
    / fossil_gain
    * 100
)

japan_fossil_sources = japan_fossil_sources.sort_values(
    "Change_2011_2012_TWh",
    ascending=False
)


print("FOSSIL GAIN: GAS vs COAL vs OTHER FOSSIL")

display(
    japan_fossil_sources[
        [
            2011,
            2012,
            2016,
            "Change_2011_2012_TWh",
            "Contribution_to_fossil_gain_%"
        ]
    ].round(2)
)

print("\n→ Tổng đóng góp của Gas, Coal và Other fossil")
print("  được đối chiếu với mức Fossil tăng thêm 149.79 TWh.")

FOSSIL GAIN: GAS vs COAL vs OTHER FOSSIL


Year,2011,2012,2016,Change_2011_2012_TWh,Contribution_to_fossil_gain_%
Electricity source,,,,,
Gas,379.83,435.49,458.43,55.66,37.16
Coal,285.07,334.30,338.59,49.23,32.87
Other fossil,162.87,207.77,97.23,44.90,29.98



→ Tổng đóng góp của Gas, Coal và Other fossil
  được đối chiếu với mức Fossil tăng thêm 149.79 TWh.


In [20]:
# ============================================================
#   Tổng hợp toàn bộ cơ chế thay đổi của sự kiện thành một bảng định lượng
# ============================================================

japan_event_summary = pd.DataFrame({
    "Metric": [
        "Total generation change",
        "Nuclear change",
        "Fossil change",
        "Renewables change",
        "Fossil replacement ratio"
    ],
    "Value": [
        total_generation_change,
        nuclear_loss,
        fossil_gain,
        renewable_change,
        replacement_ratio
    ],
    "Unit": [
        "TWh",
        "TWh",
        "TWh",
        "TWh",
        "%"
    ]
})

fossil_contribution_check = (
    japan_fossil_sources["Contribution_to_fossil_gain_%"].sum()
)

print("=" * 70)
print("NOTE 4 – JAPAN 2012 EVENT SUMMARY")
print("=" * 70)

print("\n1. Tổng hợp định lượng sự kiện:")
display(japan_event_summary.round(2))

print("\n2. Đóng góp vào mức Fossil tăng thêm:")
display(
    japan_fossil_sources[
        ["Change_2011_2012_TWh", "Contribution_to_fossil_gain_%"]
    ].round(2)
)

print(
    f"\nTổng đóng góp Gas + Coal + Other fossil: "
    f"{fossil_contribution_check:.2f}%"
)

print("\n→ Nếu giá trị xấp xỉ 100%, toàn bộ Fossil gain")
print("  đã được giải thích bởi ba nguồn Gas, Coal và Other fossil.")

NOTE 4 – JAPAN 2012 EVENT SUMMARY

1. Tổng hợp định lượng sự kiện:


,Metric,Value,Unit
0,Total generation change,2.68,TWh
1,Nuclear change,-144.94,TWh
2,Fossil change,149.79,TWh
3,Renewables change,-2.17,TWh
4,Fossil replacement ratio,103.35,%



2. Đóng góp vào mức Fossil tăng thêm:


Year,Change_2011_2012_TWh,Contribution_to_fossil_gain_%
Electricity source,,
Gas,55.66,37.16
Coal,49.23,32.87
Other fossil,44.90,29.98



Tổng đóng góp Gas + Coal + Other fossil: 100.00%

→ Nếu giá trị xấp xỉ 100%, toàn bộ Fossil gain
  đã được giải thích bởi ba nguồn Gas, Coal và Other fossil.


In [21]:
# ============================================================
#   Trong các anomaly lớn đã phát hiện, event nào có mức thay
#   đổi đáng chú ý nhất về quy mô và cơ cấu?
#
# Mục đích:
#   So sánh các candidate event và tạo cơ sở định lượng để
#   chọn Japan 2012 làm case study chính.
# ============================================================

candidate_events = [
    ("United Kingdom", 2016),
    ("Japan", 2012),
    ("Netherlands", 2021),
    ("Türkiye", 2019),
    ("United Arab Emirates", 2023)
]

event_ranking = []

for country, year in candidate_events:

    current = df[
        (df["Area"] == country) &
        (df["Area type"] == "Country or economy") &
        (df["Year"] == year) &
        (df["Electricity source"].isin(generation_sources))
    ][[
        "Electricity source",
        "Generation (TWh)",
        "Share of generation (%)"
    ]].copy()

    previous = df[
        (df["Area"] == country) &
        (df["Area type"] == "Country or economy") &
        (df["Year"] == year - 1) &
        (df["Electricity source"].isin(generation_sources))
    ][[
        "Electricity source",
        "Generation (TWh)",
        "Share of generation (%)"
    ]].copy()

    merged = current.merge(
        previous,
        on="Electricity source",
        suffixes=("_current", "_previous")
    )

    merged["share_change_pp"] = (
        merged["Share of generation (%)_current"]
        - merged["Share of generation (%)_previous"]
    )

    merged["generation_change_TWh"] = (
        merged["Generation (TWh)_current"]
        - merged["Generation (TWh)_previous"]
    )

    # Nguồn có thay đổi tỷ trọng lớn nhất về trị tuyệt đối
    max_share_change = merged.loc[
        merged["share_change_pp"].abs().idxmax()
    ]

    # Tổng sản lượng của năm event
    total_generation = df[
        (df["Area"] == country) &
        (df["Area type"] == "Country or economy") &
        (df["Year"] == year) &
        (df["Electricity source"] == "Total generation")
    ]["Generation (TWh)"].iloc[0]

    event_ranking.append({
        "Country": country,
        "Event_year": year,
        "Total_generation_TWh": total_generation,
        "Largest_source_change": max_share_change["Electricity source"],
        "Largest_share_change_pp": max_share_change["share_change_pp"],
        "Largest_generation_change_TWh": max_share_change[
            "generation_change_TWh"
        ]
    })

event_ranking = pd.DataFrame(event_ranking)

# Xếp theo độ lớn của thay đổi tỷ trọng
event_ranking = event_ranking.sort_values(
    "Largest_share_change_pp",
    key=lambda x: x.abs(),
    ascending=False
).reset_index(drop=True)

print("=" * 75)
print("CANDIDATE EVENT COMPARISON")
print("=" * 75)

display(
    event_ranking.round({
        "Total_generation_TWh": 2,
        "Largest_share_change_pp": 3,
        "Largest_generation_change_TWh": 2
    })
)

print("\n→ Bảng này dùng để giải thích tại sao Japan 2012")
print("  được chọn làm case study chính thay vì các anomaly khác.")

CANDIDATE EVENT COMPARISON


,Country,Event_year,Total_generation_TWh,Largest_source_change,Largest_share_change_pp,Largest_generation_change_TWh
0,United Kingdom,2016,339.18,Coal,-13.350,-45.21
1,Japan,2012,1106.88,Nuclear,-13.130,-144.94
2,Netherlands,2021,120.89,Gas,-12.568,-15.83
3,Türkiye,2019,300.18,Gas,-11.733,-35.21
4,United Arab Emirates,2023,167.53,Gas,-11.395,-9.15



→ Bảng này dùng để giải thích tại sao Japan 2012
  được chọn làm case study chính thay vì các anomaly khác.


In [23]:
# ============================================================
#   Các candidate event có duy trì thay đổi cơ cấu sau năm
#   xảy ra anomaly hay chỉ là biến động ngắn hạn?
#
# Mục đích:
#   So sánh persistence của các candidate bằng cách theo dõi
#   nguồn có biến động lớn nhất từ năm event đến event + 2 năm.
# ============================================================

persistence_results = []

for _, event in event_ranking.iterrows():

    country = event["Country"]
    year = int(event["Event_year"])
    source = event["Largest_source_change"]

    years_needed = [year - 1, year, year + 2]

    check = df[
        (df["Area"] == country) &
        (df["Area type"] == "Country or economy") &
        (df["Year"].isin(years_needed)) &
        (df["Electricity source"] == source)
    ][[
        "Year",
        "Share of generation (%)"
    ]].sort_values("Year")

    values = dict(
        zip(
            check["Year"],
            check["Share of generation (%)"]
        )
    )

    pre_share = values.get(year - 1)
    event_share = values.get(year)
    post_share = values.get(year + 2)

    if (
        pre_share is not None and
        event_share is not None and
        post_share is not None
    ):
        event_change = event_share - pre_share
        post_vs_pre = post_share - pre_share
        retained_change = (
            abs(post_vs_pre) / abs(event_change) * 100
            if event_change != 0
            else None
        )
    else:
        event_change = None
        post_vs_pre = None
        retained_change = None

    persistence_results.append({
        "Country": country,
        "Event_year": year,
        "Source": source,
        "Pre_event_share_%": pre_share,
        "Event_share_%": event_share,
        "Post_2yr_share_%": post_share,
        "Event_change_pp": event_change,
        "Post_vs_pre_pp": post_vs_pre,
        "Retained_change_%": retained_change
    })

persistence_comparison = pd.DataFrame(persistence_results)

print("PERSISTENCE COMPARISON OF CANDIDATE EVENTS")

display(
    persistence_comparison.round({
        "Pre_event_share_%": 3,
        "Event_share_%": 3,
        "Post_2yr_share_%": 3,
        "Event_change_pp": 3,
        "Post_vs_pre_pp": 3,
        "Retained_change_%": 1
    })
)

print("\n→ Retained_change_% cho biết sau 2 năm, mức thay đổi so với trước event còn được duy trì bao nhiêu.")
print("→ Đây là tiêu chí bổ sung để đánh giá structural break thay vì chỉ dựa vào độ lớn anomaly trong một năm.")

PERSISTENCE COMPARISON OF CANDIDATE EVENTS


,Country,Event_year,Source,Pre_event_share_%,Event_share_%,Post_2yr_share_%,Event_change_pp,Post_vs_pre_pp,Retained_change_%
0,United Kingdom,2016,Coal,22.392,9.042,5.043,-13.350,-17.349,130.0
1,Japan,2012,Nuclear,14.755,1.625,0.000,-13.130,-14.755,112.4
2,Netherlands,2021,Gas,59.462,46.894,38.421,-12.568,-21.041,167.4
3,Türkiye,2019,Gas,30.015,18.282,32.738,-11.733,2.723,23.2
4,United Arab Emirates,2023,Gas,82.081,70.686,NaN,NaN,NaN,NaN



→ Retained_change_% cho biết sau 2 năm, mức thay đổi so với trước event còn được duy trì bao nhiêu.
→ Đây là tiêu chí bổ sung để đánh giá structural break thay vì chỉ dựa vào độ lớn anomaly trong một năm.


In [25]:
# ============================================================
# NOTE 4 – CHECKPOINT
#   - Screening anomaly toàn bộ dataset
#   - Lọc anomaly theo quy mô hệ thống
#   - Shortlist candidate events
#   - So sánh các candidate
#   - Chọn Japan 2012 làm case study chính
#   - Định lượng Nuclear → Fossil replacement
#   - Kiểm tra persistence sau event
# ============================================================

note04_state = {
    "df": df,
    "generation_sources": generation_sources,
    "system_sources": system_sources,

    # Candidate comparison
    "event_ranking": event_ranking,
    "persistence_comparison": persistence_comparison,

    # Japan event analysis
    "japan_event_df": japan_event_df,
    "japan_event_generation": japan_event_generation,
    "japan_event_share": japan_event_share,
    "japan_event_share_change": japan_event_share_change,
    "japan_fossil_sources": japan_fossil_sources,
    "japan_event_summary": japan_event_summary,

    # Key quantitative results
    "nuclear_loss": nuclear_loss,
    "fossil_gain": fossil_gain,
    "renewable_change": renewable_change,
    "total_generation_change": total_generation_change,
    "replacement_ratio": replacement_ratio
}

# Nếu các bảng screening tồn tại trong notebook,
# tự động lưu thêm mà không gây lỗi.
optional_objects = [
    "anomaly_candidates",
    "filtered_candidates",
    "anomaly_df",
    "filtered_anomalies"
]

for name in optional_objects:
    if name in globals():
        note04_state[name] = globals()[name]

note04_checkpoint = (
    project_root / "checkpoints/note04_anomaly_event.pkl"
)

pd.to_pickle(note04_state, note04_checkpoint)

print("=" * 70)
print("NOTE 4 – CHECKPOINT CREATED SUCCESSFULLY")
print("=" * 70)
print("Đã lưu trạng thái Note 4 vào file .pkl.")
print()
print("Case study chính: Japan 2012")
print(f"Nuclear change: {nuclear_loss:.2f} TWh")
print(f"Fossil change: {fossil_gain:.2f} TWh")
print(f"Fossil replacement ratio: {replacement_ratio:.2f}%")
print()
print("Candidate events:", len(event_ranking))
print("Checkpoint path:")
print(note04_checkpoint)
print()
print("Các object đã lưu:", len(note04_state))

NOTE 4 – CHECKPOINT CREATED SUCCESSFULLY
Đã lưu trạng thái Note 4 vào file .pkl.

Case study chính: Japan 2012
Nuclear change: -144.94 TWh
Fossil change: 149.79 TWh
Fossil replacement ratio: 103.35%

Candidate events: 5
Checkpoint path:
d:\phan tich data\Ember_Electricity\checkpoints\note04_anomaly_event.pkl

Các object đã lưu: 17
